# 04 - Model Comparison (Decision Tree and Random Forest)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from utils import create_time_split, get_preprocessing_pipeline

df = pd.read_csv('Data/cleaned_sold.csv')
df['CloseDate'] = pd.to_datetime(df['CloseDate'])

max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(df, 'CloseDate', 12, test_start_date, max_date)

X_train = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

preprocessor = get_preprocessing_pipeline(X_train)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

Training Window (X=12 months): 2025-03-30 to 2026-03-30 | Rows: 110241
Testing Window (1 month): 2026-03-30 to 2026-04-30


### Training Decision Tree Regressor

In [9]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV


In [11]:
decision_tree = DecisionTreeRegressor(random_state=42)

param_grid = {
    'criterion': ['squared_error', 'friedman_mse'],
    'max_depth': [None, 3, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

grid_search = GridSearchCV(
    estimator=decision_tree,
    param_grid=param_grid,
    cv=3,                            
    scoring='r2', 
    n_jobs=-1,                       
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", grid_search.best_params_)
print("Best Cross-Validation Score (Negative MSE):", grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Test Set MSE:", mean_squared_error(y_test, y_pred))
print("Test Set R² Score:", r2_score(y_test, y_pred))

Fitting 3 folds for each of 270 candidates, totalling 810 fits


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:489: FitFailedWarning: 
405 fits failed out of a total of 810.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
56 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py", line 1393, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~

Best Hyperparameters: {'criterion': 'squared_error', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best Cross-Validation Score (Negative MSE): 0.7444650267244092
Test Set MSE: 86170257118.48738
Test Set R² Score: 0.7631159341730871


### Training Random Forest Regressor

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

In [17]:
rf_model = RandomForestRegressor()

rf_param_distributions = {
    'n_estimators': [50, 100],
    'criterion': ['squared_error'],
    'max_depth': [5, 10, 20, None],
    'min_samples_split':[2, 5, 10, 15],
    'min_samples_leaf':[1, 2, 4, 8],
    'max_features': ['sqrt', 'log2']
}

rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_distributions,
    n_iter=10,                       
    cv=3,                            
    scoring='r2',                     
    n_jobs=-1,                       
    random_state=42,
    verbose=1,
    error_score='raise'               
)

rf_random_search.fit(X_train, y_train)

print("Best RF Hyperparameters:", rf_random_search.best_params_)
print("Best RF Cross-Validation R² Score:", rf_random_search.best_score_)

best_rf_model = rf_random_search.best_estimator_
y_rf_pred = best_rf_model.predict(X_test)

print(f"RF Test Set MSE: {mean_squared_error(y_test, y_rf_pred):.4f}")
print(f"RF Test Set R² Score: {r2_score(y_test, y_rf_pred):.4f}")


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best RF Hyperparameters: {'n_estimators': 50, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'criterion': 'squared_error'}
Best RF Cross-Validation R² Score: 0.7964847977920108
RF Test Set MSE: 70437052625.5211
RF Test Set R² Score: 0.8064


### Comparing Current Models

In [18]:
compare_table = pd.DataFrame({
    "Model": [
        "Linear Regression", 
        "Decision Tree",
        "Random Forest"
    ],
    "Test R2": [
        0.8408, 
        0.7631,
        0.8064  
    ]
})

compare_table

,Model,Test R2
0,Linear Regression,0.8408
1,Decision Tree,0.7631
2,Random Forest,0.8064
